In [2]:
import pathlib

import numpy as np
import pandas as pd

In [42]:
data_folder = pathlib.Path("../../data/processed/biolog/")
phenotypes_folder = data_folder / "phenotypes/"
features_folder = data_folder / "features/"
features_reduced_folder = data_folder / "features_reduced/"

In [24]:
DATASETS = ["ch", "pmi", "leaf"]
FEATURES = [
    "rast",
    "kofam",
    "uniref30",
    "cluster30",
    "eggnog_kegg",
    "uniprot_trembl",
    "cluster70",
    "uniref90",
    "kofam_modules",
    "eggnog_seed",
    "cluster50",
    "cluster90",
]

In [38]:
import json
import gzip

In [39]:
from trait_prediction.utils import read_generic_features

from trait_prediction.feature_selection.reduction import (
    remove_features_with_low_variance, remove_features_with_high_correlation
)

In [40]:
import tqdm

In [41]:
pbar = tqdm.tqdm(FEATURES)
for feature in pbar:
    for dataset in DATASETS:
        pbar.set_description(f"Processing {feature} for {dataset}")
        input_file = features_folder / f"{feature}/{feature}_{dataset}_features.tsv"
        output_file = features_folder / f"{feature}/{feature}_{dataset}_features_reduced.tsv"
        if output_file.is_file():
            continue
        low_var_file = features_folder / f"{feature}/{feature}_{dataset}_low_var_features.txt"
        corr_file = features_folder / f"{feature}/{feature}_{dataset}_corr_features.json.gz"
        if feature == "kofam_modules":
            bool_conversion = False
            dtype = "float64"
            var_thres = 0.0
        else:
            bool_conversion = True
            dtype = "uint8"
            var_thres = 0.01
        features = read_generic_features(
            input_file,
            bool_conversion=bool_conversion,
            dtype=dtype
        )
        features, low_var_features = remove_features_with_low_variance(
            features, threshold=var_thres
        )
        if features.shape[1] <= 50_000:
            corr_method = "numpy"
        else:
            corr_method = "numba_parallel"
        features, correlated_features_dict = remove_features_with_high_correlation(
            features, threshold=0.95, method=corr_method
        )
        features.to_csv(output_file, sep="\t", index=True)
        with open(low_var_file, "w") as fid:
            fid.write("\n".join(low_var_features))
        with gzip.open(corr_file, "wt") as gzfile:
            json.dump(correlated_features_dict, gzfile)

Processing cluster90 for leaf: 100%|██████████| 12/12 [1:01:19<00:00, 306.66s/it]  


In [44]:
# Copy _reduced.tsv files from features folder to the features_reduced folder
import shutil

for feature in FEATURES:
    for dataset in DATASETS:
        input_file = features_folder / f"{feature}/{feature}_{dataset}_features_reduced.tsv"
        output_file = features_reduced_folder / f"{feature}/{feature}_{dataset}_features_reduced.tsv"
        print(f"Copying {feature} and {dataset} to {output_file}")
        output_folder = output_file.parent
        output_folder.mkdir(parents=True, exist_ok=True)
        if not output_file.is_file():
            shutil.copy(input_file, output_file)

Copying rast and ch to ../../data/processed/biolog/features_reduced/rast/rast_ch_features_reduced.tsv
Copying rast and pmi to ../../data/processed/biolog/features_reduced/rast/rast_pmi_features_reduced.tsv
Copying rast and leaf to ../../data/processed/biolog/features_reduced/rast/rast_leaf_features_reduced.tsv
Copying kofam and ch to ../../data/processed/biolog/features_reduced/kofam/kofam_ch_features_reduced.tsv
Copying kofam and pmi to ../../data/processed/biolog/features_reduced/kofam/kofam_pmi_features_reduced.tsv
Copying kofam and leaf to ../../data/processed/biolog/features_reduced/kofam/kofam_leaf_features_reduced.tsv
Copying uniref30 and ch to ../../data/processed/biolog/features_reduced/uniref30/uniref30_ch_features_reduced.tsv
Copying uniref30 and pmi to ../../data/processed/biolog/features_reduced/uniref30/uniref30_pmi_features_reduced.tsv
Copying uniref30 and leaf to ../../data/processed/biolog/features_reduced/uniref30/uniref30_leaf_features_reduced.tsv
Copying cluster30 a